# Three-Way Domain Model Comparison

| | `convert.py` (no arrival) | `model_clock/convert.py` | `time_precedence/convert.py` |
|---|---|---|---|
| Arrival gate | None — trains placed directly at init | `elapsed_time >= arrival(?t)` | `previous_arrived(?t)` predicate chain |
| Cost fluent | None | `elapsed_time` (clock) + `total_cost` (metric) | `total-cost` — pure metric |
| `wait` action | None | Parameterized `wait(?t)` with preconditions | Removed entirely |
| Cost increment | None | `add_increase_effect(total_cost(), N)` | `add_increase_effect(total-cost(), N)` |
| `arrive` action | None | Gated on `elapsed_time >= arrival(?t)` | Gated on `previous_arrived(?t)` |
| Planner suitability | Any planner (no timing) | Numeric planner with clock | ENHSP satisficing; CP post-processor recovers times |

In [1]:
import os, sys, subprocess, textwrap, difflib
from pathlib import Path

REPO         = Path(os.path.abspath('')).parent   # Robust-Rail-NL
PA           = REPO / 'planning-approach'
SRC          = PA / 'src' / 'convert'
BASE_SCRIPT  = SRC / 'convert.py'
CLOCK_SCRIPT = SRC / 'model_clock' / 'convert.py'
PREC_SCRIPT  = SRC / 'time_precedence' / 'convert.py'
INPUTS = REPO / 'scenario-planning-inputs' / 'Location_SimpleService'
DATA   = PA / 'data' / 'SimpleService' / 'scenario_solver_no-service'
DATA.mkdir(parents=True, exist_ok=True)

SCENARIO   = 'scenario_solver_no-service.json'
COUPLING   = 'explicit_coupling'
SUBPROBLEM = 'combined'

# Locate ENHSP jar: prefer venv-installed up_enhsp, fall back to ENHSP_JAR env var
import importlib.util
_spec = importlib.util.find_spec('up_enhsp')
if _spec:
    _enhsp_jar = Path(_spec.origin).parent / 'ENHSP' / 'enhsp.jar'
    if _enhsp_jar.exists():
        os.environ['ENHSP_JAR'] = str(_enhsp_jar)
ENHSP_JAR = os.environ.get('ENHSP_JAR', 'NOT SET')

for label, path in [
    ('baseline script',   BASE_SCRIPT),
    ('clock script',      CLOCK_SCRIPT),
    ('precedence script', PREC_SCRIPT),
    ('inputs',            INPUTS),
    ('ENHSP jar',         Path(ENHSP_JAR)),
]:
    print(f'{label}: {path}  exists={path.exists()}')

baseline script: /Users/timstols/PycharmProjects/Robust-Rail-NL/planning-approach/src/convert/convert.py  exists=True
clock script: /Users/timstols/PycharmProjects/Robust-Rail-NL/planning-approach/src/convert/model_clock/convert.py  exists=True
precedence script: /Users/timstols/PycharmProjects/Robust-Rail-NL/planning-approach/src/convert/time_precedence/convert.py  exists=True
inputs: /Users/timstols/PycharmProjects/Robust-Rail-NL/scenario-planning-inputs/Location_SimpleService  exists=True
ENHSP jar: /Users/timstols/PycharmProjects/Robust-Rail-NL/.venv/lib/python3.9/site-packages/up_enhsp/ENHSP/enhsp.jar  exists=True


## 1. Generate PDDL from all three converters

In [2]:
def run_converter(script, out_problem, out_domain):
    result = subprocess.run(
        [sys.executable, str(script),
         '-p', str(INPUTS),
         '-s', SCENARIO,
         '-o', str(out_problem),
         '-d', str(out_domain),
         '--subproblem', SUBPROBLEM,
         '--coupling-mode', COUPLING],
        capture_output=True, text=True, cwd=str(PA)
    )
    if result.returncode != 0:
        print('STDERR:', result.stderr[-500:])
        raise RuntimeError(f'{script} failed')
    label = script.parent.name + '/' + script.name if script.parent.name != 'convert' else script.name
    print(f'{label} -> {out_problem.name}, {out_domain.name}')

base_problem  = DATA / 'nb_base_problem.pddl'
base_domain   = DATA / 'nb_base_domain.pddl'
clock_problem = DATA / 'nb_clock_problem.pddl'
clock_domain  = DATA / 'nb_clock_domain.pddl'
prec_problem  = DATA / 'nb_prec_problem.pddl'
prec_domain   = DATA / 'nb_prec_domain.pddl'

run_converter(BASE_SCRIPT,  base_problem,  base_domain)
run_converter(CLOCK_SCRIPT, clock_problem, clock_domain)
run_converter(PREC_SCRIPT,  prec_problem,  prec_domain)

convert.py -> nb_base_problem.pddl, nb_base_domain.pddl
model_clock/convert.py -> nb_clock_problem.pddl, nb_clock_domain.pddl
time_precedence/convert.py -> nb_prec_problem.pddl, nb_prec_domain.pddl


## 2. Domain structure comparison

In [3]:
import re

def extract_actions(pddl_text):
    return re.findall(r':action\s+(\S+)', pddl_text)

def extract_functions(pddl_text):
    m = re.search(r'\(:functions([^)]+(?:\([^)]*\)[^)]*)*?)\)', pddl_text, re.S)
    if not m:
        return []
    return [tok.strip() for tok in m.group(1).split('\n') if tok.strip()]

base_dom_txt  = base_domain.read_text()
clock_dom_txt = clock_domain.read_text()
prec_dom_txt  = prec_domain.read_text()

base_actions  = extract_actions(base_dom_txt)
clock_actions = extract_actions(clock_dom_txt)
prec_actions  = extract_actions(prec_dom_txt)

print('Actions — convert.py (no arrival):       ', base_actions)
print('Actions — model_clock/convert.py:        ', clock_actions)
print('Actions — time_precedence/convert.py:    ', prec_actions)
print()
print('Added in model_clock vs base:      ', set(clock_actions) - set(base_actions))
print('Removed in time_precedence vs clock:', set(clock_actions) - set(prec_actions))
print('Added in time_precedence vs clock:  ', set(prec_actions) - set(clock_actions))

Actions — convert.py (no arrival):        ['start_move', 'end_move', 'move_aside_empty', 'move_aside_occupied', 'move_bside_empty', 'move_bside_occupied', 'depart_aside', 'depart_bside', 'park', 'uncouple', 'couple_two_units', 'couple_two_units_same_train', 'match']
Actions — model_clock/convert.py:         ['start_move', 'end_move', 'move_aside_empty', 'move_aside_occupied', 'move_bside_empty', 'move_bside_occupied', 'depart_aside', 'depart_bside', 'park', 'wait', 'arrive', 'uncouple', 'couple_two_units', 'couple_two_units_same_train', 'match']
Actions — time_precedence/convert.py:     ['start_move', 'end_move', 'move_aside_empty', 'move_aside_occupied', 'move_bside_empty', 'move_bside_occupied', 'depart_aside', 'depart_bside', 'park', 'arrive', 'uncouple', 'couple_two_units', 'couple_two_units_same_train', 'match']

Added in model_clock vs base:       {'wait', 'arrive'}
Removed in time_precedence vs clock: {'wait'}
Added in time_precedence vs clock:   set()


In [4]:
base_funcs  = extract_functions(base_dom_txt)
clock_funcs = extract_functions(clock_dom_txt)
prec_funcs  = extract_functions(prec_dom_txt)

print('Numeric functions — convert.py (no arrival):')
for f in base_funcs: print(' ', f)
print()
print('Numeric functions — model_clock/convert.py:')
for f in clock_funcs: print(' ', f)
print()
print('Numeric functions — time_precedence/convert.py:')
for f in prec_funcs: print(' ', f)

Numeric functions — convert.py (no arrival):
  (arrival ?train - arrivaltrain

Numeric functions — model_clock/convert.py:
  (arrival ?train - arrivaltrain

Numeric functions — time_precedence/convert.py:
  (arrival ?train - arrivaltrain


In [5]:
def extract_action_block(pddl_text, action_name):
    pattern = rf'\(:action {action_name}.*?(?=\(:action|\)\s*$)'
    m = re.search(pattern, pddl_text, re.S)
    return m.group(0).strip() if m else '(not found)'

for action in ['wait', 'arrive']:
    for label, dom in [
        ('convert.py (no arrival)', base_dom_txt),
        ('model_clock/convert.py',  clock_dom_txt),
        ('time_precedence/convert.py', prec_dom_txt),
    ]:
        print(f'=== {action} — {label} ===')
        print(textwrap.fill(extract_action_block(dom, action), width=110, subsequent_indent='  '))
        print()

=== wait — convert.py (no arrival) ===
(not found)

=== wait — model_clock/convert.py ===
(:action wait   :parameters ( ?t - arrivaltrain)   :precondition (and (not (has_arrived ?t)) (< (elapsed_time)
  (arrival ?t)))   :effect (and (increase (total_cost) 300) (increase (elapsed_time) 300)))

=== wait — time_precedence/convert.py ===
(not found)

=== arrive — convert.py (no arrival) ===
(not found)

=== arrive — model_clock/convert.py ===
(:action arrive   :parameters ( ?t - arrivaltrain ?l - trackpart)   :precondition (and (not (has_arrived ?t))
  (entry_track_of ?t ?l) (<= (arrival ?t) (elapsed_time)))   :effect (and (has_arrived ?t) (at ?t ?l) (assign
  (aside_distance ?t) (bstack_distance ?l)) (assign (bstack_distance ?l) (+ (train_length ?t) (bstack_distance
  ?l))) (assign (number_of_trains_on_track ?l) (+ 1 (number_of_trains_on_track ?l)))))

=== arrive — time_precedence/convert.py ===
(:action arrive   :parameters ( ?t - arrivaltrain ?l - trackpart)   :precondition (and (not (h

## 3. Problem file — arrival-related facts

In [6]:
base_prob_txt  = base_problem.read_text()
clock_prob_txt = clock_problem.read_text()
prec_prob_txt  = prec_problem.read_text()

ARRIVAL_KEYS = ('total_cost', 'elapsed_time', 'total-cost', 'previous_arrived', 'arrival_immediately_before', 'arrival', 'has_arrived')

for label, txt in [
    ('convert.py (no arrival)', base_prob_txt),
    ('model_clock/convert.py',  clock_prob_txt),
    ('time_precedence/convert.py', prec_prob_txt),
]:
    print(f'{label} — timing-related init lines:')
    for line in txt.splitlines():
        if any(k in line for k in ARRIVAL_KEYS):
            print(' ', line.strip())
    print()

convert.py (no arrival) — timing-related init lines:
  train11111 train33333 - arrivaltrain
  (= (arrival train11111) 1500)
  (= (arrival train33333) 1900)

model_clock/convert.py — timing-related init lines:
  train11111 train33333 - arrivaltrain
  (= (arrival train11111) 1500)
  (= (arrival train33333) 1900)
  (= (total_cost) 0)
  (= (elapsed_time) 0)
  (:metric minimize (total_cost))

time_precedence/convert.py — timing-related init lines:
  train11111 train33333 - arrivaltrain
  (= (arrival train11111) 1500)
  (= (arrival train33333) 1900)
  (previous_arrived train11111)
  (arrival_immediately_before train11111 train33333)
  (= (total-cost) 0)
  (:metric minimize (total-cost))



## 4. PDDL diffs — domain files

In [7]:
for from_label, from_txt, to_label, to_txt in [
    ('convert.py', base_dom_txt, 'model_clock/convert.py', clock_dom_txt),
    ('model_clock/convert.py', clock_dom_txt, 'time_precedence/convert.py', prec_dom_txt),
]:
    diff = list(difflib.unified_diff(
        from_txt.splitlines(keepends=True),
        to_txt.splitlines(keepends=True),
        fromfile=from_label,
        tofile=to_label,
        n=0
    ))
    print(f'--- diff: {from_label}  →  {to_label} ---')
    print(''.join(diff[:120]))
    print()

--- diff: convert.py  →  model_clock/convert.py ---
--- convert.py
+++ model_clock/convert.py
@@ -14,0 +15,2 @@
+             (has_arrived ?train - arrivaltrain)
+             (entry_track_of ?train - arrivaltrain ?trackpart - trackpart)
@@ -44,0 +47,2 @@
+             (total_cost)
+             (elapsed_time)
@@ -48 +52 @@
-  :precondition (and (not (allowed_to_move ?t)) (< (concurrent_movements) 1))
+  :precondition (and (not (allowed_to_move ?t)) (< (concurrent_movements) 1) (has_arrived ?t))
@@ -57 +61 @@
-  :effect (and (assign (number_of_trains_on_track ?l_from) (- (number_of_trains_on_track ?l_from) 1)) (assign (number_of_trains_on_track ?l_to) 1) (assign (aside_distance ?t) 0) (assign (astack_distance ?l_from) (+ (train_length ?t) (astack_distance ?l_from))) (assign (astack_distance ?l_to) 0) (assign (bstack_distance ?l_to) (train_length ?t)) (at ?t ?l_to) (not (at ?t ?l_from))))
+  :effect (and (assign (number_of_trains_on_track ?l_from) (- (number_of_trains_on_track ?l_from) 

## 5. Benchmark — ENHSP (WA* hmax)

In [9]:
import time

PLANNER = str(PA / 'src' / 'plan' / 'planner.jl')

def run_planner(domain_file, problem_file, label):
    env = os.environ.copy()
    start = time.monotonic()
    result = subprocess.run(
        ['julia', f'--project={PA}', PLANNER,
         str(domain_file), str(problem_file), 'enhsp'],
        capture_output=True, text=True, env=env
    )
    elapsed = time.monotonic() - start
    plan_file = Path(str(problem_file).replace('.pddl', '.plan'))
    plan_len  = len([l for l in plan_file.read_text().splitlines() if l.strip()]) if plan_file.exists() else -1
    found = 'Solved' in result.stdout
    print(f'{label:56s}  {elapsed:6.1f}s  plan={plan_len:3d}  found={found}')
    if not found:
        print('  STDOUT:', result.stdout[-300:])
        print('  STDERR:', result.stderr[-300:])
    return elapsed, plan_len, found

print(f'{"Model":56s}  {"Time":>6s}  {"Plan":>7s}  Found')
print('-' * 80)
run_planner(base_domain,  base_problem,  'convert.py         (no arrival modeling)')
# run_planner(clock_domain, clock_problem, 'model_clock        (elapsed_time clock)')
run_planner(prec_domain,  prec_problem,  'time_precedence    (arrival predicates)')

Model                                                       Time     Plan  Found
--------------------------------------------------------------------------------
convert.py         (no arrival modeling)                     3.4s  plan= 52  found=True
time_precedence    (arrival predicates)                      2.7s  plan= 26  found=True


(2.694828333000004, 26, True)

## 6. Plan comparison

In [10]:
ACTION_COSTS = {
    'move_aside_empty': 300, 'move_aside_occupied': 300,
    'move_bside_empty': 300, 'move_bside_occupied': 300,
    'wait': 300, 'uncouple': 120,
    'couple_two_units': 180, 'couple_two_units_same_train': 180,
}

def plan_cost(plan_file):
    total = 0
    for line in Path(plan_file).read_text().splitlines():
        name = line.strip().split('(')[0].lower()
        total += ACTION_COSTS.get(name, 0)
    return total

base_plan  = str(base_problem).replace('.pddl', '.plan')
clock_plan = str(clock_problem).replace('.pddl', '.plan')
prec_plan  = str(prec_problem).replace('.pddl', '.plan')

for label, plan_path in [
    ('convert.py         ', base_plan),
    ('model_clock        ', clock_plan),
    ('time_precedence    ', prec_plan),
]:
    p = Path(plan_path)
    if p.exists():
        lines = [l.strip() for l in p.read_text().splitlines() if l.strip()]
        cost = plan_cost(plan_path)
        print(f'{label}  cost={cost:5d}s  steps={len(lines)}')
        for i, l in enumerate(lines, 1):
            print(f'  {i:2}. {l}')
    else:
        print(f'{label}  (no plan file)')
    print()

convert.py           cost=    0s  steps=52
   1. (start_move train33333)
   2. (move_aside_empty train33333 bumper_10 rail_5)
   3. (move_aside_empty train33333 rail_5 switch_21)
   4. (move_aside_empty train33333 switch_21 rail_1)
   5. (move_aside_empty train33333 rail_1 switch_20)
   6. (move_aside_empty train33333 switch_20 rail_3)
   7. (end_move train33333 rail_3)
   8. (start_move train11111)
   9. (move_aside_empty train11111 bumper_13 rail_4)
  10. (move_aside_empty train11111 rail_4 switch_21)
  11. (move_bside_empty train11111 switch_21 rail_5)
  12. (move_bside_empty train11111 rail_5 bumper_10)
  13. (move_aside_empty train11111 bumper_10 rail_5)
  14. (move_aside_empty train11111 rail_5 switch_21)
  15. (move_aside_empty train11111 switch_21 rail_1)
  16. (move_bside_empty train11111 rail_1 switch_21)
  17. (move_bside_empty train11111 switch_21 rail_4)
  18. (end_move train11111 rail_4)
  19. (start_move train33333)
  20. (match unit2422 request11112_slot0)
  21. (move_b

## 7. Design difference summary

### convert.py (no arrival modeling)

Trains are placed on their entry tracks at init time. No `arrive` action, no cost metric. The planner
treats all trains as immediately available and optimises only physical movement.

### model_clock/convert.py

Adds an `elapsed_time` clock fluent and a parameterized `wait(?t)` action. `arrive` is gated on
`elapsed_time >= arrival(?t)`. The planner must advance the clock with `wait` steps before each train
can enter, which explodes the branching factor.

### time_precedence/convert.py

Drops the clock entirely. Arrival order is encoded as boolean predicates
(`previous_arrived`, `arrival_immediately_before`). Each `arrive` action unlocks the next train via
a `forall` conditional effect. `total-cost` is a pure metric with `add_increase_effect`.

### Progression summary

| | convert.py | model_clock | time_precedence |
|---|---|---|---|
| Arrival modeled | No | Yes (numeric clock) | Yes (predicate chain) |
| `wait` action | None | Parameterized, guarded | Removed |
| Clock states | None | Every wait/move = new state | None |
| Cost heuristic | N/A | Noisy (wait + move mixed) | Clean (move only) |
| Planner suitability | Any | Numeric | ENHSP / cost-aware |